In [3]:
from bs4 import BeautifulSoup
import json
import re

def clean_wiki_text(text):
    # Loại bỏ các tham chiếu chú thích như [1], [ghi chú 1]
    text = re.sub(r'\[.*?\]', '', text)
    # Loại bỏ ký tự khoảng trắng không ngắt và khoảng trắng thừa
    return text.strip().replace('\xa0', ' ')

def extract_full_data(input_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')

    danh_sach = []
    # Tìm bảng danh sách Trạng nguyên có class 'wikitable'
    table = soup.find('table', class_='wikitable')

    if table:
        rows = table.find_all('tr')
        # Lấy tiêu đề để xác định đúng thứ tự cột
        header_cells = [clean_wiki_text(th.get_text()) for th in rows[0].find_all(['th', 'td'])]
        
        # Ánh xạ cột dựa trên tiêu đề thực tế trong file HTML
        # Các tiêu đề thực tế: "Thứ tự", "Tên", "Năm sinh năm mất", "Quê", "Năm đỗ Trạng nguyên", "Đời vua", "Ghi chú"
        col_map = {
            "name": -1, 
            "years": -1, 
            "home": -1, 
            "exam_year": -1, 
            "king": -1
        }
        
        for i, header in enumerate(header_cells):
            h_lower = header.lower()
            if h_lower == "tên": col_map["name"] = i
            elif "năm sinh" in h_lower: col_map["years"] = i
            elif "quê" in h_lower: col_map["home"] = i
            elif "năm đỗ" in h_lower: col_map["exam_year"] = i
            elif "vua" in h_lower: col_map["king"] = i

        # Duyệt qua các hàng dữ liệu
        for row in rows[1:]:
            cols = row.find_all(['td', 'th'])
            # Bỏ qua nếu hàng không đủ số lượng cột cần thiết
            if len(cols) <= max(col_map.values()):
                continue

            # Trích xuất dữ liệu
            name = clean_wiki_text(cols[col_map["name"]].get_text()) if col_map["name"] != -1 else ""
            
            # Chỉ xử lý nếu lấy được tên nhân vật hợp lệ
            if name and not name.isdigit():
                item = {
                    "nhan_vat": name,
                    "nam_sinh_mat": clean_wiki_text(cols[col_map["years"]].get_text()) if col_map["years"] != -1 else "Không rõ",
                    "que_quan": clean_wiki_text(cols[col_map["home"]].get_text()) if col_map["home"] != -1 else "Không rõ",
                    "nam_do": clean_wiki_text(cols[col_map["exam_year"]].get_text()) if col_map["exam_year"] != -1 else "Không rõ",
                    "doi_vua": clean_wiki_text(cols[col_map["king"]].get_text()) if col_map["king"] != -1 else "Không rõ"
                }
                danh_sach.append(item)

    return danh_sach

# Thực thi trích xuất từ file của bạn
results = extract_full_data(r'D:\MScThi\12_02_2026\first_trial.txt')

# Lưu kết quả ra JSON
with open('data_trang_nguyen.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Đã trích xuất thành công {len(results)} nhân vật.")

Đã trích xuất thành công 55 nhân vật.


**DOWNLOAD WIKI EACH CHARACTER**

In [5]:
import json
import requests
import os
import time
import re
from urllib.parse import quote

class WikiAgent:
    def __init__(self, storage_folder="wiki_html_storage"):
        self.storage_folder = storage_folder
        self.base_url = "https://vi.wikipedia.org/wiki/"
        self.headers = {
            'User-Agent': 'WikiCrawlerAgent/1.0 (contact: your@email.com)'
        }
        if not os.path.exists(self.storage_folder):
            os.makedirs(self.storage_folder)

    def clean_filename(self, name):
        """Tạo tên file an toàn cho hệ thống."""
        return re.sub(r'[\\/*?:"<>|]', "", name).replace(" ", "_")

    def fetch_and_save(self, character_name):
        """Agent truy cập, tải và lưu trữ HTML."""
        # Xử lý các tên có nhiều lựa chọn (ví dụ: Trịnh Tuệ / Trịnh Huệ)
        target_name = character_name.split('/')[0].strip()
        file_path = os.path.join(self.storage_folder, f"{self.clean_filename(target_name)}.txt")

        if os.path.exists(file_path):
            return f"SKIP: {target_name} đã tồn tại."

        # Agent thực hiện truy vấn
        url = f"{self.base_url}{quote(target_name.replace(' ', '_'))}"
        try:
            response = requests.get(url, headers=self.headers, timeout=15)
            
            if response.status_code == 200:
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(response.text)
                return f"SUCCESS: Đã lưu {target_name}"
            elif response.status_code == 404:
                return f"FAILED: Không tìm thấy trang cho {target_name}"
            else:
                return f"ERROR: Mã lỗi {response.status_code} cho {target_name}"
        
        except Exception as e:
            return f"EXCEPTION: {str(e)}"

def run_agent_workflow(input_json):
    # Khởi tạo Agent
    agent = WikiAgent()

    # Đọc dữ liệu từ file JSON
    if not os.path.exists(input_json):
        print(f"Lỗi: Không tìm thấy file {input_json}")
        return

    with open(input_json, 'r', encoding='utf-8') as f:
        characters = json.load(f)

    print(f"Wiki Agent đang xử lý {len(characters)} nhân vật...")

    # Agent bắt đầu làm việc
    for item in characters:
        name = item.get('nhan_vat')
        if name:
            log = agent.fetch_and_save(name)
            print(log)
            time.sleep(0.5) # Độ trễ để đảm bảo an toàn

if __name__ == "__main__":
    # Chạy quy trình với file JSON của bạn
    run_agent_workflow('data_trang_nguyen.json')

Wiki Agent đang xử lý 55 nhân vật...
SUCCESS: Đã lưu Khương Công Phụ
SUCCESS: Đã lưu Lê Văn Thịnh
SUCCESS: Đã lưu Mạc Hiển Tích
SUCCESS: Đã lưu Bùi Quốc Khái
SUCCESS: Đã lưu Nguyễn Công Bình
SUCCESS: Đã lưu Trương Hanh
SUCCESS: Đã lưu Lưu Miễn
SUCCESS: Đã lưu Nguyễn Quan Quang
SUCCESS: Đã lưu Nguyễn Hiền
SUCCESS: Đã lưu Trần Quốc Lặc
SUCCESS: Đã lưu Trương Xán
SUCCESS: Đã lưu Trần Cố
SUCCESS: Đã lưu Bạch Liêu
SUCCESS: Đã lưu Lý Đạo Tái
SUCCESS: Đã lưu Đào Tiêu
SUCCESS: Đã lưu Mạc Đĩnh Chi
SUCCESS: Đã lưu Đào Sư Tích
SUCCESS: Đã lưu Lưu Thúc Kiệm
SUCCESS: Đã lưu Nguyễn Trực
SUCCESS: Đã lưu Nguyễn Nghiêu Tư
SUCCESS: Đã lưu Lương Thế Vinh
SUCCESS: Đã lưu Vũ Kiệt
SUCCESS: Đã lưu Vũ Tuấn Chiêu
SUCCESS: Đã lưu Phạm Đôn Lễ
SUCCESS: Đã lưu Nguyễn Quang Bật
SUCCESS: Đã lưu Trần Sùng Dĩnh
SUCCESS: Đã lưu Vũ Duệ
FAILED: Không tìm thấy trang cho Vũ Tích (Vũ Dương)
SUCCESS: Đã lưu Nghiêm Hoản
SUCCESS: Đã lưu Đỗ Lý Khiêm
SUCCESS: Đã lưu Lê Ích Mộc
SUCCESS: Đã lưu Lê Nại
SUCCESS: Đã lưu Nguyễn Giản T

**EXTRACT EACH CHARACTER INFO**

In [8]:
import os
import json
import re
from bs4 import BeautifulSoup

class BulkWikiParser:
    def __init__(self, input_folder="wiki_html_storage", output_file="trang_nguyen_full_database.json"):
        self.input_folder = input_folder
        self.output_file = output_file

    def clean_wiki_text(self, text):
        """Làm sạch triệt để văn bản, xóa chú thích và chuẩn hóa khoảng trắng."""
        if not text: return ""
        # Xóa các tham chiếu chú thích: [1], [2], [a], [ghi chú 1]
        text = re.sub(r'\[[^\]]*\]', '', text)
        # Thay thế ký tự lạ và xuống dòng
        text = text.replace('\xa0', ' ').replace('\n', ' ')
        # Loại bỏ khoảng trắng thừa
        return re.sub(r'\s+', ' ', text).strip()

    def parse_file(self, file_path):
        """Trích xuất chi tiết từ một file HTML Wikipedia."""
        with open(file_path, 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')

        # 1. Tên nhân vật
        title_tag = soup.find('h1', id='first_heading') or soup.find('h1', id='firstHeading')
        name = title_tag.get_text(strip=True) if title_tag else "Unknown"

        content_body = soup.find('div', class_='mw-parser-output')
        summary = ""
        biography_sections = {}
        
        if content_body:
            # 2. Trích xuất Tóm tắt (trước Mục lục hoặc Header đầu tiên)
            summary_parts = []
            for elem in content_body.find_all(['p', 'ul'], recursive=False):
                if elem.name == 'p' and len(elem.get_text(strip=True)) > 20:
                    summary_parts.append(self.clean_wiki_text(elem.get_text()))
                if elem.name == 'div' and ('toc' in elem.get('id', '') or 'toc' in elem.get('class', [])):
                    break
                if elem.name in ['h2', 'h3']:
                    break
            summary = " ".join(summary_parts)

            # 3. Trích xuất Plain Text theo từng Section (Sự nghiệp, Cuộc đời...)
            current_header = "Thông tin chung"
            for elem in content_body.find_all(['h2', 'h3', 'p', 'ul']):
                if elem.name in ['h2', 'h3']:
                    header_text = self.clean_wiki_text(elem.get_text().replace('sửa', '').replace('mã nguồn', ''))
                    # Bỏ qua các mục phụ lục cuối trang
                    if header_text in ["Ghi chú", "Tham khảo", "Xem thêm", "Liên kết ngoài", "Nguồn"]:
                        current_header = None
                    else:
                        current_header = header_text
                elif current_header and elem.name in ['p', 'ul']:
                    text = self.clean_wiki_text(elem.get_text())
                    if text and len(text) > 10:
                        biography_sections.setdefault(current_header, []).append(text)

        # 4. Trích xuất Infobox (Dữ liệu đặc tả)
        infobox = {}
        info_table = soup.find('table', class_='infobox')
        if info_table:
            for row in info_table.find_all('tr'):
                label = row.find(['th', 'td'], class_='infobox-label') or row.find('th')
                value = row.find(['td'], class_='infobox-data') or row.find('td')
                if label and value and label != value:
                    key = self.clean_wiki_text(label.get_text())
                    val = self.clean_wiki_text(value.get_text())
                    if key: infobox[key] = val

        # 5. Phân loại (Categories)
        categories = []
        cat_links = soup.find('div', id='mw-normal-catlinks')
        if cat_links:
            categories = [self.clean_wiki_text(li.get_text()) for li in cat_links.find_all('li')]

        return {
            "nhan_vat": name,
            "tom_tat": summary,
            "chi_tiet_theo_muc": {k: " ".join(v) for k, v in biography_sections.items() if v},
            "du_lieu_infobox": infobox,
            "phan_loai_wiki": categories
        }

    def run(self):
        """Duyệt toàn bộ thư mục và gộp dữ liệu."""
        all_data = []
        if not os.path.exists(self.input_folder):
            print(f"Lỗi: Thư mục {self.input_folder} không tồn tại.")
            return

        files = [f for f in os.listdir(self.input_folder) if f.endswith('.txt')]
        print(f"Đang bắt đầu xử lý hàng loạt {len(files)} file...")

        for idx, filename in enumerate(files):
            file_path = os.path.join(self.input_folder, filename)
            try:
                data = self.parse_file(file_path)
                all_data.append(data)
                print(f"[{idx+1}/{len(files)}] √ Đã xử lý: {data['nhan_vat']}")
            except Exception as e:
                print(f"[{idx+1}/{len(files)}] x Lỗi tại file {filename}: {e}")

        # Lưu kết quả cuối cùng
        with open(self.output_file, 'w', encoding='utf-8') as out_f:
            json.dump(all_data, out_f, ensure_ascii=False, indent=4)
        
        print(f"\n--- XỬ LÝ HOÀN TẤT ---")
        print(f"Cơ sở dữ liệu đã được lưu tại: {os.path.abspath(self.output_file)}")

if __name__ == "__main__":
    parser = BulkWikiParser()
    parser.run()

Đang bắt đầu xử lý hàng loạt 54 file...
[1/54] √ Đã xử lý: Bùi Quốc Khái
[2/54] √ Đã xử lý: Bạch Liêu
[3/54] √ Đã xử lý: Dương Phúc Tư
[4/54] √ Đã xử lý: Giáp Hải
[5/54] √ Đã xử lý: Hoàng Nghĩa Phú
[6/54] √ Đã xử lý: Hoàng Văn Tán
[7/54] √ Đã xử lý: Khương Công Phụ
[8/54] √ Đã xử lý: Lê Nại
[9/54] √ Đã xử lý: Lê Văn Thịnh
[10/54] √ Đã xử lý: Lê Ích Mộc
[11/54] √ Đã xử lý: Huyền Quang
[12/54] √ Đã xử lý: Lưu Danh Công
[13/54] √ Đã xử lý: Lưu Miễn (định hướng)
[14/54] √ Đã xử lý: Lưu Thúc Kiệm
[15/54] √ Đã xử lý: Lương Thế Vinh
[16/54] √ Đã xử lý: Mạc Hiển Tích
[17/54] √ Đã xử lý: Mạc Đĩnh Chi
[18/54] √ Đã xử lý: Nghiêm Hoản
[19/54] √ Đã xử lý: Nguyễn Bỉnh Khiêm
[20/54] √ Đã xử lý: Nguyễn Công Bình
[21/54] √ Đã xử lý: Nguyễn Giản Thanh
[22/54] √ Đã xử lý: Nguyễn Hiền
[23/54] √ Đã xử lý: Nguyễn Kỳ
[24/54] √ Đã xử lý: Nguyễn Lượng Thái
[25/54] √ Đã xử lý: Nguyễn Nghiêu Tư
[26/54] √ Đã xử lý: Nguyễn Quang Bật
[27/54] √ Đã xử lý: Nguyễn Quan Quang
[28/54] √ Đã xử lý: Nguyễn Quốc Trinh
[29/54